In [5]:
from dotenv import load_dotenv
from pathlib import Path  
envPath = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

print(envPath)

if envPath.exists():
    load_dotenv(dotenv_path= envPath, override=True)
    print("env loaded")
else:
    print("Env file missing")

IMAGE_TAG = "Latest"
REPO_NAME = "node-server-less-prac"

c:\Users\VijayKumar\Desktop\Projects\personal\python-practice-combined\env\dev.aws.env
env loaded


In [6]:
from botocore.exceptions import ClientError
import boto3
import os
import docker

endpoint_url = os.getenv("AWS_URL")
region = os.getenv("REGION")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

print(os.getenv("APP_NAME"), endpoint_url, region, aws_access_key_id, aws_secret_access_key)

AWS PRACTICE http://localhost:4566 us-east-1 vijay vijay


In [3]:
docker_client = docker.from_env()

images_list = docker_client.images.list()

for image in images_list:
    print(image.short_id.split(":")[1], image.tags[0] )
    # for item in dir(image):
    #     if not item.startswith('_'):  
    #         print(item)
    # print("\n============\n")

58a654a551bc node-severless-ecr-ecs-app:latest
cc1eea824f30 ministackorg/ministack:latest
93cd31802f0f docker-traderbuddy:latest
50e1c02fcdcd docker-tb-backend:latest
4b496a938116 davireis/stackport:latest
8efc0bf21599 searxng/searxng:latest


In [4]:
ecr_client= boto3.client("ecr", endpoint_url=endpoint_url, region_name= region,
    aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)

In [5]:
from IPython.core.magics import execution
from math import exp
"""PUSH IMAGE TO ECR"""
try:
    print("Creating ECR Repository :: ")


    ecr_repo =ecr_client.create_repository(
        repositoryName=REPO_NAME,
    )

except ClientError as ce :
    print("=====>>>>>>>>>>>\n")
    print(ce)
except Exception as e:
    print("============>\n")
    print(e)

Creating ECR Repository :: 
=====>>>>>>>>>>>

An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'node-server-less-prac' already exists


In [6]:
### PUSH IMAGE to ECR REPO###
try:
    repo =  ecr_client.describe_repositories(repositoryNames=[REPO_NAME])
    repository_uri = repo["repositories"][0]["repositoryUri"]

    # ### pushing image to ECR
    print("Pushing image to AWS ECR (this may take a few minutes)...")
    # # Reference implementation strategy: https://github.com/AlexIoannides/py-docker-aws-example-project/blob/master/deploy_to_aws.py
    push_logs = docker_client.images.push(repository_uri, tag=IMAGE_TAG, stream=True, decode=True)
    #
    for log in push_logs:
        if "status" in log:
            status = log.get("status")
            progress = log.get("progress", "")
            print(f"{status} {progress}")

    print("✨Successfully created repository and pushed Docker image to Amazon ECR!")

except ClientError as ce :
    print("=====>>>>>>>>>>>\n")
    print(ce)
except Exception as e:
    print("============>\n")
    print(e)

Pushing image to AWS ECR (this may take a few minutes)...
The push refers to repository [000000000000.dkr.ecr.us-east-1.amazonaws.com/node-server-less-prac] 
✨Successfully created repository and pushed Docker image to Amazon ECR!


In [7]:
# Initialize AWS Clients
ec2_client = boto3.client("ec2", endpoint_url=endpoint_url, region_name= region,
    aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)
ecs_client = boto3.client("ecs", endpoint_url=endpoint_url, region_name= region,
    aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)
iam_client = boto3.client("iam", endpoint_url=endpoint_url, region_name= region,
    aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)

In [11]:
try:
    print("\nCreating ECS Cluster...")
    cluster_name = "sample_cluster"
    # Create ECS cluster
    cluster = ecs_client.create_cluster(clusterName=cluster_name)
    print(f"\nECS Cluster Created named {cluster_name}...")

    # Creaet ECS task

    print("\nRegistering ECS Task Definition...")
    task_family = f"{cluster_name}-task"
    task_response = ecs_client.register_task_definition(
        family=task_family,
        cpu="256",
        memory="512",
        network_mode="family",
        requiresCompatibilities=["FARGATE"],
        executionRoleArn=role_arn,
        containerDefinitions=[
                {
                    "name": "app-container",
                    "image": "000000000000.dkr.ecr.us-east-1.amazonaws.com/node-server-less-prac",
                    "essential": True,
                    "portMappings": [
                        {
                            "containerPort": 3000,
                            "hostPort": 3000,
                            "protocol": "tcp"
                        }
                    ]
                }
            ]
        
    )
    task_def_arn = task_response["taskDefinition"]["taskDefinitionArn"]
    print(f"✅ Registered Task Definition: {task_def_arn}")

    pass
except ClientError as ce :
    print("=====>>>>>>>>>>>\n")
    print(ce)
except Exception as e:
    print("============>\n")
    print(e)


Creating ECS Cluster...

ECS Cluster Created named sample_cluster...

Registering ECS Task Definition...
============>

Parameter validation failed:
Unknown parameter in input: "network_mode", must be one of: family, taskRoleArn, executionRoleArn, networkMode, containerDefinitions, volumes, placementConstraints, requiresCompatibilities, cpu, memory, tags, pidMode, ipcMode, proxyConfiguration, inferenceAccelerators, ephemeralStorage, runtimePlatform, enableFaultInjection
